# Diffusion model — recover δ from raw CB data

Training set: `data_diffusion/` — 2000 memory runs, each at a continuous random
δ ∈ [0.02, 0.25]. Goal: read one run's raw f_(P,m,l) and predict its δ **with error bars**.

This notebook: (1) load + eyeball the data, then (2) build the model (later).

In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

In [8]:
# load the 2000 parquet files

DATA_DIR = Path("data_diffusion")
files = sorted(DATA_DIR.glob("*.parquet"))
df_all = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
print(f"loaded files: {len(files)} with rows: {len(df_all)}")

# fleeting look
print("δ range :", round(df_all.param_value.min(), 4), "→", round(df_all.param_value.max(), 4))
print("unique δ:", df_all.param_value.nunique())           # expect 2000 (one per run)
print("Paulis  :", sorted(df_all.P.unique()), "| m:", sorted(df_all.m.unique()), "| l:", df_all.l.nunique())
display(df_all.describe())
display(df_all.tail())



loaded files: 2000 with rows: 1080000
δ range : 0.02 → 0.2499
unique δ: 2000
Paulis  : ['X', 'Y', 'Z'] | m: [2, 4, 6, 8, 12, 16] | l: 30


,param_value,seed,m,l,f_pml,n_plus,n_minus,pauli_sign
count,1.080000e+06,1.080000e+06,1.080000e+06,1.080000e+06,1.080000e+06,1.080000e+06,1.080000e+06,1.080000e+06
mean,1.347503e-01,9.995000e+02,8.000000e+00,1.450000e+01,6.158239e-01,5.119772e+02,5.120228e+02,-2.148148e-04
std,6.686932e-02,5.773505e+02,4.760954e+00,8.655445e+00,4.004342e-01,3.760977e+02,3.760977e+02,1.000000e+00
min,2.004370e-02,0.000000e+00,2.000000e+00,0.000000e+00,-1.000000e+00,0.000000e+00,0.000000e+00,-1.000000e+00
25%,7.680562e-02,4.997500e+02,4.000000e+00,7.000000e+00,4.296875e-01,1.300000e+02,1.300000e+02,-1.000000e+00
50%,1.343580e-01,9.995000e+02,7.000000e+00,1.450000e+01,7.421875e-01,5.120000e+02,5.120000e+02,-1.000000e+00
75%,1.939882e-01,1.499250e+03,1.200000e+01,2.200000e+01,9.433594e-01,8.940000e+02,8.940000e+02,1.000000e+00
max,2.498853e-01,1.999000e+03,1.600000e+01,2.900000e+01,1.000000e+00,1.024000e+03,1.024000e+03,1.000000e+00


,noisetype,param,param_value,seed,P,m,l,f_pml,n_plus,n_minus,pauli_sign,twirl_seq
1079995,memory,delta,0.093958,1999,Z,16,25,0.814453,95,929,-1,"[I, Z, X, I, Y, Z, I, Z, X, Y, Z, X, Z, Y, Z, ..."
1079996,memory,delta,0.093958,1999,Z,16,26,-0.167969,598,426,-1,"[X, Z, Y, Z, Z, X, Y, Y, Y, Y, Y, X, X, I, Y, ..."
1079997,memory,delta,0.093958,1999,Z,16,27,0.033203,495,529,-1,"[X, I, Y, I, I, X, Z, I, I, Z, I, Z, Z, Z, I, ..."
1079998,memory,delta,0.093958,1999,Z,16,28,0.945312,28,996,-1,"[X, Z, Y, Z, Y, Z, Z, I, Z, X, Y, Z, Y, Y, Z, ..."
1079999,memory,delta,0.093958,1999,Z,16,29,-0.136719,442,582,1,"[Z, I, Z, Y, I, Z, X, I, I, I, I, X, Z, I, Y, ..."


In [9]:
ex_keys = ["seed", "param_value"]
df_agg_mean_var = df_all.groupby(ex_keys + ["P", "m"])["f_pml"].agg(["mean", "var"]).reset_index()
print(df_agg_mean_var.shape)
display(df_agg_mean_var.head())

df_feature_view = df_agg_mean_var.pivot_table(index=ex_keys, columns=["P", "m"], values=["mean", "var"])
df_feature_view.columns = [f"{stat}_{P}{m}" for stat, P, m in df_feature_view.columns]
df_feature_view = df_feature_view.reset_index()
print(df_feature_view.shape)
display(df_feature_view.head())


(36000, 6)


,seed,param_value,P,m,mean,var
0,0,0.166501,X,2,0.885677,0.011255
1,0,0.166501,X,4,0.758008,0.050881
2,0,0.166501,X,6,0.686784,0.087442
3,0,0.166501,X,8,0.643685,0.094608
4,0,0.166501,X,12,0.490560,0.119027


(2000, 38)


,seed,param_value,mean_X2,mean_X4,mean_X6,mean_X8,mean_X12,mean_X16,mean_Y2,mean_Y4,...,var_Y6,var_Y8,var_Y12,var_Y16,var_Z2,var_Z4,var_Z6,var_Z8,var_Z12,var_Z16
0,0,0.166501,0.885677,0.758008,0.686784,0.643685,0.490560,0.469792,0.731771,0.767383,...,0.072790,0.133871,0.202031,0.144874,0.071937,0.093232,0.091134,0.099584,0.196820,0.242025
1,1,0.082051,0.975911,0.933398,0.929948,0.860482,0.874740,0.757878,0.907096,0.823958,...,0.034499,0.030003,0.046618,0.153003,0.008886,0.027701,0.031453,0.037493,0.063274,0.182765
2,2,0.029424,0.997331,0.994076,0.988932,0.984831,0.961784,0.956706,0.986784,0.969857,...,0.004860,0.004488,0.010261,0.010492,0.000341,0.000670,0.001423,0.004224,0.005538,0.005585
3,3,0.023801,0.998047,0.993945,0.992318,0.985872,0.988542,0.976823,0.992122,0.971810,...,0.000902,0.002011,0.001886,0.004112,0.000132,0.000872,0.001243,0.001360,0.002103,0.006593
4,4,0.207052,0.841862,0.617188,0.608138,0.547201,0.256315,0.128190,0.713932,0.541992,...,0.125531,0.173927,0.095763,0.218162,0.067575,0.078727,0.128648,0.167616,0.205652,0.200387
